# Reading EHR data

EHR data reaches you in many shapes: a wide spreadsheet, a long table of events, a MEDS or FHIR export, or an OMOP database.
ehrdata reads all of them into the same {class}`~ehrdata.EHRData` object of patients × variables × time intervals.
This tutorial follows a few synthetic ICU stays through every reader and ends with the example datasets that ship with ehrdata.

:::{admonition} Which reader?
:class: tip

A table with one column per variable and timepoint goes through {func}`~ehrdata.io.from_pandas`, a table with one row per event through {func}`~ehrdata.io.from_events`.
MEDS datasets, FHIR exports, and OMOP databases have their own readers, which bin events in the same way.
Read raw data once and save the result as `.h5ed` or `.zarr`.
:::

In [1]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import ehrdata as ed

workdir = Path(tempfile.mkdtemp())

## A wide table

Spreadsheet exports often have one row per patient and one column per variable and timepoint, such as `heart_rate_t_0`.

In [2]:
df_wide = pd.DataFrame(
    {
        "patient_id": ["1", "2", "3"],
        "sex": ["female", "male", "female"],
        "age": [67, 54, 81],
        "heart_rate_t_0": [88.0, 72.0, 104.0],
        "heart_rate_t_1": [80.0, np.nan, 97.0],
        "systolic_bp_t_0": [132.0, 118.0, 96.0],
        "systolic_bp_t_1": [128.0, 121.0, np.nan],
    }
)
df_wide

,patient_id,sex,age,heart_rate_t_0,heart_rate_t_1,systolic_bp_t_0,systolic_bp_t_1
0,1,female,67,88.0,80.0,132.0,128.0
1,2,male,54,72.0,NaN,118.0,121.0
2,3,female,81,104.0,97.0,96.0,NaN


{func}`~ehrdata.io.from_pandas` with `format="wide"` moves the suffixes to the time axis and keeps the patient-level columns in `obs`.

In [3]:
edata = ed.io.from_pandas(
    df_wide, format="wide", columns_obs_only=["patient_id", "sex", "age"], index_column="patient_id"
)
edata

EHRData object with n_obs × n_vars × n_t = 3 × 2 × 2
    obs: 'sex', 'age'
    shape of .X: (3, 2, 2)

In [4]:
edata.X

array([[[ 88.,  80.],
        [132., 128.]],

       [[ 72.,  nan],
        [118., 121.]],

       [[104.,  97.],
        [ 96.,  nan]]])

:::{note}
The suffixes are sorted lexicographically, so zero-pad them (`t_01`, ..., `t_10`) if there are ten or more timepoints.
:::

## A long table of events

Most EHR systems store one row per event instead: a subject, a time, a code, and optionally a value.
Here, heart rate and creatinine carry values, while the diuretic furosemide is only recorded as given.

In [5]:
events = pd.DataFrame(
    {
        "subject_id": [1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3],
        "time": pd.to_datetime(
            [
                "2024-03-01 08:10",
                "2024-03-01 09:05",
                "2024-03-01 09:40",
                "2024-03-01 13:30",
                "2024-03-01 14:00",
                "2024-03-02 22:15",
                "2024-03-03 01:00",
                "2024-03-03 02:30",
                "2024-03-05 06:00",
                "2024-03-05 06:20",
                "2024-03-05 12:45",
            ]
        ),
        "code": ["heart_rate", "heart_rate", "furosemide", "heart_rate", "creatinine"]
        + ["heart_rate", "creatinine", "heart_rate"]
        + ["heart_rate", "furosemide", "heart_rate"],
        "numeric_value": [88, 95, None, 79, 1.4, 72, 0.9, 70, 104, None, 97],
    }
)
events

,subject_id,time,code,numeric_value
0,1,2024-03-01 08:10:00,heart_rate,88.0
1,1,2024-03-01 09:05:00,heart_rate,95.0
2,1,2024-03-01 09:40:00,furosemide,NaN
3,1,2024-03-01 13:30:00,heart_rate,79.0
4,1,2024-03-01 14:00:00,creatinine,1.4
5,2,2024-03-02 22:15:00,heart_rate,72.0
6,2,2024-03-03 01:00:00,creatinine,0.9
7,2,2024-03-03 02:30:00,heart_rate,70.0
8,3,2024-03-05 06:00:00,heart_rate,104.0
9,3,2024-03-05 06:20:00,furosemide,NaN


{func}`~ehrdata.io.from_events` turns the codes into variables and divides the time of each subject into intervals of equal length, by default one hour from the first event.
Codes without values, like furosemide, count the events per interval.

In [6]:
edata = ed.io.from_events(events)
edata

EHRData object with n_obs × n_vars × n_t = 3 × 3 × 7
    obs: 'anchor_time'
    var: 'n_events'
    tem: '0', '1', '2', '3', '4', '5', '6'
    shape of .X: (3, 3, 7)

In [7]:
edata.tem

,interval_start_offset,interval_end_offset
interval_step,,
0,0 days 00:00:00,0 days 01:00:00
1,0 days 01:00:00,0 days 02:00:00
2,0 days 02:00:00,0 days 03:00:00
3,0 days 03:00:00,0 days 04:00:00
4,0 days 04:00:00,0 days 05:00:00
5,0 days 05:00:00,0 days 06:00:00
6,0 days 06:00:00,0 days 07:00:00


The time axis usually starts at a clinical anchor, such as the admission, rather than at the first event.
Pass the anchor as a column of `obs` and choose the interval length, the number of intervals, and how values within one interval are aggregated.

In [8]:
admissions = pd.DataFrame(
    {
        "admission_time": pd.to_datetime(["2024-03-01 08:00", "2024-03-02 22:00", "2024-03-05 06:00"]),
        "age": [67, 54, 81],
    },
    index=[1, 2, 3],
)
edata = ed.io.from_events(
    events,
    obs=admissions,
    anchor="admission_time",
    interval_length_number=6,
    interval_length_unit="h",
    num_intervals=2,
    aggregation_strategy="mean",
)
edata.X

array([[[         nan,   1.4       ],
        [  1.        ,          nan],
        [ 87.33333333,          nan]],

       [[  0.9       ,          nan],
        [         nan,          nan],
        [ 71.        ,          nan]],

       [[         nan,          nan],
        [  1.        ,          nan],
        [104.        ,  97.        ]]])

Patient 1 now has the mean of three heart rates in the first six hours and the creatinine in the second.

:::{admonition} Which aggregation?
:class: tip

Keep the default `"last"` for vitals and labs that describe the current state, and use `"mean"`, `"min"`, or `"max"` to summarize an interval.
:::

:::{note}
{func}`~ehrdata.io.from_events` also reads pyarrow tables, DuckDB relations, and parquet files, including glob patterns such as `"events/*.parquet"`.
Pass `sparse=True` if most codes are missing in most intervals.
:::

## MEDS

The [Medical Event Data Standard (MEDS)](https://medical-event-data-standard.github.io) stores such event tables as parquet files, together with code metadata and the split of every subject.
{func}`~ehrdata.io.write_meds` writes every value as an event at the start of its interval.

In [9]:
edata.obs["split"] = ["train", "train", "held_out"]
ed.io.write_meds(edata, workdir / "meds", anchor="admission_time")
sorted(path.relative_to(workdir).as_posix() for path in (workdir / "meds").rglob("*.*"))

['meds/data/0.parquet',
 'meds/metadata/codes.parquet',
 'meds/metadata/dataset.json',
 'meds/metadata/subject_splits.parquet']

:::{warning}
Pass the column with the anchor times to {func}`~ehrdata.io.write_meds`, here `anchor="admission_time"`.
Without it, the events are dated from 1970-01-01.
:::

{func}`~ehrdata.io.read_meds` bins the events with {func}`~ehrdata.io.from_events` and can select subjects by their split.
Static events without a time, here the age, become columns of `obs`.

In [10]:
meds = ed.io.read_meds(workdir / "meds", split="train", interval_length_number=6, interval_length_unit="h")
meds

EHRData object with n_obs × n_vars × n_t = 2 × 3 × 2
    obs: 'split', 'age', 'anchor_time'
    var: 'n_events'
    tem: '0', '1'
    uns: 'meds_static_codes'
    shape of .X: (2, 3, 2)

In [11]:
meds.obs

,split,age,anchor_time
1,train,67.0,2024-03-01 08:00:00
2,train,54.0,2024-03-02 22:00:00


:::{tip}
{func}`~ehrdata.io.read_meds` starts the intervals at the first event of each subject.
To start them at the admission instead, pass the parquet files in `data/` and an `obs` with the admission times to {func}`~ehrdata.io.from_events`.
:::

## FHIR

[FHIR R4](https://hl7.org/fhir/R4) exchanges every clinical fact as a JSON resource, in Bundles or as Bulk Data `.ndjson` exports.
We write a small Bundle with two patients, LOINC-coded observations, ICD-10-CM diagnoses, and an ATC-coded medication.

In [12]:
def observation(patient, code, display, value, unit, time):
    return {
        "resourceType": "Observation",
        "status": "final",
        "subject": {"reference": f"Patient/{patient}"},
        "code": {"coding": [{"system": "http://loinc.org", "code": code, "display": display}]},
        "valueQuantity": {"value": value, "unit": unit},
        "effectiveDateTime": time,
    }


def condition(patient, code, display, time):
    return {
        "resourceType": "Condition",
        "subject": {"reference": f"Patient/{patient}"},
        "code": {"coding": [{"system": "http://hl7.org/fhir/sid/icd-10-cm", "code": code, "display": display}]},
        "onsetDateTime": time,
    }


resources = [
    {"resourceType": "Patient", "id": "1", "gender": "female", "birthDate": "1957-04-12"},
    {"resourceType": "Patient", "id": "2", "gender": "male", "birthDate": "1970-09-30"},
    observation("1", "8867-4", "Heart rate", 88, "/min", "2024-03-01T08:10:00Z"),
    observation("1", "8867-4", "Heart rate", 79, "/min", "2024-03-01T13:30:00Z"),
    observation("1", "2160-0", "Creatinine", 1.4, "mg/dL", "2024-03-01T14:00:00Z"),
    observation("2", "8867-4", "Heart rate", 72, "/min", "2024-03-02T22:15:00Z"),
    condition("1", "I21.0", "STEMI of anterior wall", "2024-03-01T08:30:00Z"),
    condition("2", "I21.4", "NSTEMI", "2024-03-02T23:00:00Z"),
    condition("2", "E11.9", "Type 2 diabetes mellitus without complications", "2024-03-02T23:00:00Z"),
    {
        "resourceType": "MedicationRequest",
        "status": "active",
        "subject": {"reference": "Patient/1"},
        "medicationCodeableConcept": {
            "coding": [{"system": "http://www.whocc.no/atc", "code": "C07AB02", "display": "metoprolol"}]
        },
        "authoredOn": "2024-03-01T09:00:00Z",
    },
]
with open(workdir / "bundle.json", "w") as f:
    json.dump({"resourceType": "Bundle", "type": "collection", "entry": [{"resource": r} for r in resources]}, f)

{func}`~ehrdata.io.read_fhir` names every variable by its vocabulary and code, such as `LOINC/8867-4`, and stores the patient demographics in `obs`.

In [13]:
fhir = ed.io.read_fhir(workdir / "bundle.json", interval_length_number=6, interval_length_unit="h")
fhir

EHRData object with n_obs × n_vars × n_t = 2 × 6 × 1
    obs: 'gender', 'birth_date', 'deceased', 'deceased_time', 'anchor_time'
    var: 'n_events', 'description', 'unit'
    tem: '0'
    shape of .X: (2, 6, 1)

In [14]:
fhir.var

,n_events,description,unit
ATC/C07AB02,1,metoprolol,None
ICD10CM/E11.9,1,Type 2 diabetes mellitus without complications,None
ICD10CM/I21.0,1,STEMI of anterior wall,None
ICD10CM/I21.4,1,NSTEMI,None
LOINC/2160-0,1,Creatinine,mg/dL
LOINC/8867-4,3,Heart rate,/min


:::{note}
For a Bulk Data export with one `.ndjson` file per resource type, pass the export directory.
Pass `anchor="birth_date"` to start the intervals at birth.
:::

## Annotating codes

Because the variables are codes, {func}`~ehrdata.annotate_codes` adds their vocabulary and hierarchy, such as the ICD-10 category of a diagnosis or the ATC levels of a drug.

In [15]:
ed.annotate_codes(fhir)
fhir.var[["vocabulary", "code", "icd_category", "atc_level_3"]]

,vocabulary,code,icd_category,atc_level_3
ATC/C07AB02,ATC,C07AB02,<NA>,ATC/C07A
ICD10CM/E11.9,ICD10CM,E11.9,ICD10CM/E11,<NA>
ICD10CM/I21.0,ICD10CM,I21.0,ICD10CM/I21,<NA>
ICD10CM/I21.4,ICD10CM,I21.4,ICD10CM/I21,<NA>
LOINC/2160-0,LOINC,2160-0,<NA>,<NA>
LOINC/8867-4,LOINC,8867-4,<NA>,<NA>


{func}`~ehrdata.aggregate_codes` merges the variables of one category, here both myocardial infarctions into `ICD10CM/I21`.

In [16]:
by_category = ed.aggregate_codes(fhir, "icd_category", aggregation_strategy="max")
by_category.var_names.tolist()

['ATC/C07AB02', 'ICD10CM/E11', 'ICD10CM/I21', 'LOINC/2160-0', 'LOINC/8867-4']

:::{seealso}
Descriptions of LOINC or SNOMED CT codes and their mapping to ICD require an OMOP vocabulary database, passed as `backend_handle` to {func}`~ehrdata.annotate_codes`.
:::

## Saving

Reading raw exports takes time, so save the result in one of the native formats of ehrdata.
{func}`~ehrdata.io.write_h5ed` writes a single HDF5 file.

In [17]:
ed.io.write_h5ed(meds, workdir / "cohort.h5ed")
ed.io.read_h5ed(workdir / "cohort.h5ed")

• Harmonizing missing values of X


EHRData object with n_obs × n_vars × n_t = 2 × 3 × 2
    obs: 'split', 'age', 'anchor_time'
    var: 'n_events'
    tem: '0', '1'
    uns: 'meds_static_codes'
    shape of .X: (2, 3, 2)

{func}`~ehrdata.io.write_zarr` writes a zarr store, a directory of chunked arrays.

In [18]:
ed.io.write_zarr(meds, workdir / "cohort.zarr")
ed.io.read_zarr(workdir / "cohort.zarr")

• Harmonizing missing values of X


EHRData object with n_obs × n_vars × n_t = 2 × 3 × 2
    obs: 'split', 'age', 'anchor_time'
    var: 'n_events'
    tem: '0', '1'
    uns: 'meds_static_codes'
    shape of .X: (2, 3, 2)

:::{admonition} Which format?
:class: tip

Use `.h5ed` for one portable file.
Use `.zarr` for large data and cloud object storage.
:::

:::{warning}
Neither format stores datetime columns, such as `admission_time` above, so convert them with `.astype(str)` before saving.
:::

## Example datasets

To try out methods without your own data, `ed.dt` provides example datasets.
{func}`~ehrdata.dt.ehrdata_blobs` simulates time series of patient groups without any download.

In [19]:
blobs = ed.dt.ehrdata_blobs(n_observations=100, base_timepoints=24, missing_values=0.1)
blobs

EHRData object with n_obs × n_vars × n_t = 100 × 11 × 24
    obs: 'cluster'
    tem: '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23'
    shape of .X: (100, 11, 24)

The other datasets are downloaded once and cached in `ehrapy_data/`.
{func}`~ehrdata.dt.mimic_2` is a 300 kB table of 1,776 ICU patients {cite}`critical2016secondary`.

In [20]:
mimic2 = ed.dt.mimic_2()
mimic2

EHRData object with n_obs × n_vars × n_t = 1776 × 46 × 1
    shape of .X: (1776, 46)

:::{note}
{func}`~ehrdata.dt.physionet2012` and {func}`~ehrdata.dt.physionet2019` load ICU time series, and {func}`~ehrdata.dt.mimic_iv_omop`, {func}`~ehrdata.dt.gibleed_omop`, and {func}`~ehrdata.dt.synthea27nj_omop` load OMOP databases.
:::

## OMOP

:::{seealso}
Data in the OMOP Common Data Model is read with the functions in `ed.io.omop`, as shown in the [OMOP Introduction](omop_intro).
:::

## Next Tutorial

Continue with **[Real Dataset Example: PhysioNet 2019](real_dataset_example_physionet2019)** to work with real ICU time series.